# TGLF surrogate: the five-network ensemble

This notebook covers the baseline and surrogate requirements in assignment
Steps 2–3. The selected GELU/ReLU surrogate averages five dense neural networks trained
with seeds **42–46**. It uses the fixed **3,496 training / 750 validation** split.
The mean and linear baselines provide the required reference for evaluation.

The cells load the completed models. Training instructions are at the end.
[1. Prepare data](01_prepare_data.ipynb) · [2. NN ensemble](02_nn_ensemble.ipynb) · [3. Evaluation](03_evaluation.ipynb)

In [1]:
from pathlib import Path
import json
import sys
import joblib
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

candidates = [q for p in (Path.cwd(), *Path.cwd().parents) for q in (p, p / "assignments/A2")]
A2_DIR = next(p for p in candidates if (p / "nn_ensemble.py").is_file() and (p / "models").is_dir())
sys.path.insert(0, str(A2_DIR))
import torch
from nn_ensemble import TARGETS, SEEDS, digest, load_data, load_ensemble, predict_ensemble, metric_table
from evaluate_surrogate import baseline_prediction

torch.set_num_threads(2)
MODELS = A2_DIR / "models"
RESULTS = A2_DIR / "results"
data = load_data(A2_DIR / "prepared")
members = load_ensemble(MODELS)
baselines = joblib.load(MODELS / "baseline_models.joblib")
print(f"Training: {len(data['ids']['train']):,}; validation: {len(data['ids']['validation']):,}")

Training: 3,496; validation: 750


## Model and training settings

Each network uses **11 → 64 → 64 → 8 → 3** units, with GELU in the two 64-unit
layers and ReLU in the eight-unit layer. Inputs and model targets are standardized
using training rows only. The targets are log10 ion heat, log10 electron heat,
and signed electron particle flux.

Training uses AdamW with batch size 256 and weight decay 0.0001. Each member
receives 30 warmup epochs at learning rate 0.001, then up to 120 epochs at the
selected rate 0.003. The saved checkpoint minimizes average validation RMSE
normalized by each target's training standard deviation; early stopping allows
25 epochs without improvement.

For each prediction, convert each member's heat outputs back with `10**value`,
then average the five predictions in original flux units. All five weights are
**0.2**. The ensemble reduces sensitivity to random initialization and minibatch
order; its member spread is not a calibrated uncertainty interval.

In [2]:
member_details = pd.DataFrame([
    {"seed": saved["config"]["seed"], "selected_epoch_after_warmup": saved["best_epoch"],
     "parameters": sum(p.numel() for p in model.parameters())}
    for model, saved in members
])
display(member_details.set_index("seed"))
print(f"Total ensemble parameters: {member_details.parameters.sum():,}")

,selected_epoch_after_warmup,parameters
seed,,
42,40,5475
43,60,5475
44,60,5475
45,75,5475
46,95,5475


Total ensemble parameters: 27,375


## Validation choice

We compared GELU/tanh, ReLU/tanh, GELU/ReLU, and ReLU/ReLU in the separate
[activation notebook](04_activation_comparison.ipynb). All four used five matching
seeds, initial weights, data splits, and training settings. **GELU/ReLU** had the
lowest mean normalized validation RMSE, **0.1806**, versus **0.1887** for GELU/tanh.
We promoted its exact five checkpoints without refitting or changing their weights.

The learning rate 0.003 came from the earlier GELU/tanh search and stayed fixed
through this comparison. We have not separately tuned every activation combination.
The test set was inspected in earlier work; the activation ranking used validation
error, and subsequent test comparisons are exploratory. SAT3 cases did not select
the activation or ensemble weights.

In [3]:
search = pd.read_csv(MODELS / "activation_selection.csv")
display(search.sort_values("validation_nrmse").round(6))
assert search.loc[search.validation_nrmse.idxmin(), "variant"] == "gelu_relu"
assert all(saved["config"]["hidden_activation"] == "gelu" and
           saved["config"]["bottleneck_activation"] == "relu" for _, saved in members)


,variant,validation_nrmse
0,gelu_relu,0.180636
1,gelu_tanh,0.188674
2,relu_tanh,0.193534
3,relu_relu,0.210451


## Required baselines and validation accuracy

The **mean baseline** returns each training target's arithmetic mean in original
units. **Linear regression** uses standardized inputs and fits the same log10
heat and signed particle targets as the NN. Its heat predictions are converted
back before evaluation. Neither baseline uses regularization or interactions.

The table compares all three models on the same validation cases. Final test
accuracy, timing, learning curves, and SAT3 results are in the evaluation notebook.

In [4]:
features = pd.read_csv(A2_DIR / "prepared/features.csv", index_col="case", float_precision="round_trip")
validation_ids = data["ids"]["validation"]
predictions = {name: baseline_prediction(baselines, name, features.loc[validation_ids]) for name in ["mean", "linear"]}
predictions["nn_ensemble_equal5"] = predict_ensemble(members, data["X_validation"])["prediction"]
validation = pd.concat([metric_table(data["raw_validation"], {"prediction": pred}, name) for name, pred in predictions.items()], ignore_index=True)
saved = pd.read_csv(RESULTS / "validation_metrics.csv")
keys = ["model", "target"]
np.testing.assert_allclose(validation.set_index(keys)[["rmse_gb", "r2"]].sort_index(), saved.set_index(keys)[["rmse_gb", "r2"]].sort_index(), rtol=5e-4, atol=5e-4)
labels = {"mean": "Mean baseline", "linear": "Linear regression", "nn_ensemble_equal5": "Five-NN ensemble"}
report = saved.assign(model=saved.model.map(labels)).pivot(index="model", columns="target", values=["rmse_gb", "r2"])
display(report.reindex(index=list(labels.values())).round(4))
print("The saved models reproduce the validation results. These are the selected GELU/ReLU checkpoints.")

rmse_gb                        r2                
target             Gamma_e      Q_e      Q_i Gamma_e     Q_e     Q_i
model                                                               
Mean baseline      24.1381  66.4643  88.2693 -0.0000 -0.0003 -0.0002
Linear regression  15.8892  36.7097  49.1173  0.5667  0.6949  0.6903
Five-NN ensemble    4.6066  13.1589  15.7963  0.9636  0.9608  0.9680

The saved models reproduce the validation results. These are the selected GELU/ReLU checkpoints.


## Reproduce training

The NN-only implementation is [nn_ensemble.py](nn_ensemble.py). Install the
packages in [requirements.txt](requirements.txt). After syncing this A2 folder
to the DSMLP assignment directory, run inside the course compute container:

```sh
cd ~/private/assignments/DSC180A-A2-gelu-relu
OMP_NUM_THREADS=4 OPENBLAS_NUM_THREADS=4 ~/private/.venvs/tglf-a2-py312/bin/python nn_ensemble.py --threads 4 --output-dir models_repeat
```

This fits both baselines and five NNs using the fixed settings. A new output
folder preserves the selected models. Use `--smoke` for a small local check.
Continue to [evaluation](03_evaluation.ipynb).